# Improved Visualisations — Wet vs Dry Driver Analysis

Five improvements over `driver_ranking_vamsi.ipynb`:

| # | What | Why it's better |
|---|------|------------------|
| 1 | **Percentile-rank heatmap** | MinMax lets one outlier compress everyone else; percentile rank distributes scores fairly |
| 2 | **Bubble scatter** | Adds rainy-race count as bubble size; labels on hover only, no overlap |
| 3 | **Strip + error-bar chart** | Shows mean *and* spread of finish positions — a consistent P5 is very different from a volatile P5 |
| 4 | **Lap-trajectory lines** | Shows *how* top rain specialists build their position over a race, vs their dry average |
| 5 | **Weather correlation heatmap** | Tells us which weather variables (humidity, track temp, wind…) most strongly predict lap time delta |


In [3]:
import pandas as pd
import numpy as np
import altair as alt

df = pd.read_pickle('../data/f1_lap_weather_data.pkl')

time_cols = ['LapTime','Sector1Time','Sector2Time','Sector3Time']
for col in time_cols:
    df[col] = pd.to_timedelta(df[col]).dt.total_seconds()

df['Position'] = pd.to_numeric(df['Position'], errors='coerce')
df['RaceID']   = df['EventName'] + ' ' + df['Year'].astype(str)

clean = df[
    (df['TrackStatus'] == 1)
    & (df['IsPitLap']     == False)
    & (df['IsTerminalLap'] == False)
].copy()

print(f'Clean laps: {len(clean):,}  |  Races: {clean["RaceID"].nunique()}')

Clean laps: 156,243  |  Races: 172


In [4]:
def race_stats(subset):
    rows = []
    for race_id in subset['RaceID'].unique():
        rd = subset[subset['RaceID'] == race_id]
        for driver in rd['Driver'].unique():
            dd = rd[rd['Driver'] == driver]
            pos = dd['Position'].dropna()
            if len(pos) == 0:
                continue
            first_lap = dd[dd['LapNumber'] == dd['LapNumber'].min()]
            grid  = first_lap['Position'].iloc[0] if pd.notna(first_lap['Position'].iloc[0]) else np.nan
            finish = pos.iloc[-1]
            max_laps = rd['LapNumber'].max()
            rows.append({
                'Driver': driver, 'Team': dd['Team'].iloc[0],
                'GridPosition': grid, 'FinishingPosition': finish,
                'PositionsGained': (grid - finish) if pd.notna(grid) else np.nan,
                'LapsLed': (dd['Position'] == 1.0).sum(),
                'PositionStd': pos.std(),
                'DNF': dd['LapNumber'].max() < (max_laps - 2)
            })
    return pd.DataFrame(rows)

rain_races = race_stats(clean[clean['Rainfall'] == True])
dry_races  = race_stats(clean[clean['Rainfall'] == False])

def agg_driver(df_r, prefix):
    return (
        df_r.groupby('Driver')
        .agg(
            Team=('Team','first'),
            RacesEntered=('FinishingPosition','count'),
            AvgFinish=('FinishingPosition','mean'),
            StdFinish=('FinishingPosition','std'),
            AvgGained=('PositionsGained','mean'),
            DNFRate=('DNF', lambda x: x.sum()/len(x)),
            Wins=('FinishingPosition', lambda x: (x==1).sum()),
            Podiums=('FinishingPosition', lambda x: (x<=3).sum()),
            TotalLapsLed=('LapsLed','sum'),
            AvgPosStd=('PositionStd','mean')
        )
        .add_prefix(prefix)
        .rename(columns={f'{prefix}Team': 'Team'})
        .reset_index()
    )

rain_agg = agg_driver(rain_races, 'Rain_')
dry_agg  = agg_driver(dry_races,  'Dry_')

comp = rain_agg.merge(dry_agg, on=['Driver','Team'])
comp['FinishDelta'] = comp['Rain_AvgFinish'] - comp['Dry_AvgFinish']  # neg = better in rain
comp['GainedDelta'] = comp['Rain_AvgGained'] - comp['Dry_AvgGained']

MIN_RAIN = 5
comp = comp[comp['Rain_RacesEntered'] >= MIN_RAIN].copy()
print(f'Drivers in comparison: {len(comp)}')
comp.head(3)

Drivers in comparison: 25


,Driver,Team,Rain_RacesEntered,Rain_AvgFinish,Rain_StdFinish,Rain_AvgGained,Rain_DNFRate,Rain_Wins,Rain_Podiums,Rain_TotalLapsLed,...,Dry_AvgFinish,Dry_StdFinish,Dry_AvgGained,Dry_DNFRate,Dry_Wins,Dry_Podiums,Dry_TotalLapsLed,Dry_AvgPosStd,FinishDelta,GainedDelta
0,ALB,Racing Bulls,12,11.083333,4.999242,1.416667,0.000000,0,0,0,...,10.853659,4.029976,1.227642,0.130081,0,2,1,1.854539,0.229675,0.189024
1,ALO,McLaren,19,9.052632,4.235743,0.000000,0.157895,0,2,0,...,9.200000,3.804119,0.276923,0.146154,0,8,4,1.612069,-0.147368,-0.276923
4,BOT,Mercedes,19,9.157895,4.810442,1.210526,0.210526,0,3,0,...,8.375000,5.826003,0.868056,0.111111,8,46,405,1.317012,0.782895,0.342471


---
## Chart 1 — Percentile-Rank Heatmap

**Improvement over the original:** The previous heatmap used MinMax normalisation — one outlier driver stretches the scale and squashes everyone else into a narrow band. Percentile rank assigns each driver their relative position within the field (0 = worst, 1 = best), so the full colour range is always used and comparisons are fair regardless of outliers.

Eight metrics are shown: four for rain, four for dry. Rows sorted by overall rain score (sum of rain percentile ranks).

In [5]:
hm_metrics = [
    ('Rain Finish Pos',  'Rain_AvgFinish',  True),
    ('Rain Consistency', 'Rain_AvgPosStd',  True),   # lower std = more consistent = better
    ('Rain Pos Gained',  'Rain_AvgGained',  False),
    ('Rain DNF Rate',    'Rain_DNFRate',    True),
    ('Dry Finish Pos',   'Dry_AvgFinish',   True),
    ('Dry Consistency',  'Dry_AvgPosStd',   True),
    ('Dry Pos Gained',   'Dry_AvgGained',   False),
    ('Dry DNF Rate',     'Dry_DNFRate',     True),
]

hm_rows = []
for label, col, invert in hm_metrics:
    # percentile rank: higher = better
    ranked = comp[col].rank(pct=True)
    if invert:
        ranked = 1 - ranked
    for driver, score, raw in zip(comp['Driver'], ranked, comp[col]):
        hm_rows.append({
            'Driver': driver, 'Metric': label,
            'Score': round(score, 3), 'RawValue': round(raw, 3)
        })

hm_df = pd.DataFrame(hm_rows)

# Sort drivers by sum of rain percentile scores
rain_score = (
    hm_df[hm_df['Metric'].str.startswith('Rain')]
    .groupby('Driver')['Score'].sum()
    .sort_values(ascending=False)
)
driver_order = rain_score.index.tolist()
metric_order = [m[0] for m in hm_metrics]

chart1 = alt.Chart(hm_df).mark_rect().encode(
    x=alt.X('Metric:N', sort=metric_order, title=None,
            axis=alt.Axis(labelAngle=-30, labelFontSize=11)),
    y=alt.Y('Driver:N', sort=driver_order, title='Driver (best rain performers at top)',
            axis=alt.Axis(labelFontSize=11)),
    color=alt.Color('Score:Q',
                    scale=alt.Scale(scheme='redyellowgreen', domain=[0, 1]),
                    legend=alt.Legend(title='Percentile\nRank')),
    tooltip=[
        'Driver', 'Metric',
        alt.Tooltip('Score:Q', format='.0%', title='Percentile'),
        alt.Tooltip('RawValue:Q', format='.3f', title='Raw Value')
    ]
).properties(
    title=alt.TitleParams(
        'Driver Performance Heatmap — Percentile Rank (green = best in field)',
        fontSize=14
    ),
    width=520, height=460
)

chart1

alt.Chart(...)

---
## Chart 2 — Bubble Scatter: Rain vs Dry Finish Position

**Improvement over the original:** The original scatter labelled every point with the driver's name, causing severe overlap. This version encodes three variables at once — x (dry finish), y (rain finish), and **bubble size (number of rainy races)** — so drivers with more wet-weather experience carry visual weight. Labels appear on hover only.

Points **below** the diagonal finish better in rain; points **above** finish worse.

In [10]:
diag = pd.DataFrame({'x': [1, 20], 'y': [1, 20]})
ref  = alt.Chart(diag).mark_line(
    color='gray', strokeDash=[5, 4], opacity=0.55
).encode(x='x:Q', y='y:Q')

bubbles = alt.Chart(comp).mark_circle(opacity=0.8).encode(
    x=alt.X('Dry_AvgFinish:Q',  title='Dry Avg Finish Position',
            scale=alt.Scale(domain=[1, 17])),
    y=alt.Y('Rain_AvgFinish:Q', title='Rain Avg Finish Position',
            scale=alt.Scale(domain=[1, 17])),
    size=alt.Size('Rain_RacesEntered:Q',
                  scale=alt.Scale(range=[60, 700]),
                  legend=alt.Legend(title='Rainy Races')),
    color=alt.Color('Team:N', legend=alt.Legend(title='Team')),
    tooltip=[
        'Driver', 'Team',
        alt.Tooltip('Rain_AvgFinish:Q',    format='.2f', title='Rain Avg Finish'),
        alt.Tooltip('Dry_AvgFinish:Q',     format='.2f', title='Dry Avg Finish'),
        alt.Tooltip('FinishDelta:Q',       format='.2f', title='Delta (Rain−Dry)'),
        alt.Tooltip('Rain_RacesEntered:Q', title='Rainy Races')
    ]
)

chart2 = (ref + bubbles).properties(
    title=alt.TitleParams(
        'Rain vs Dry Avg Finish — bubble size = rainy races entered',
        fontSize=13
    ),
    width=500, height=450
)

chart2

alt.LayerChart(...)

---
## Chart 3 — Mean ± Std Dev: Finish Position Spread

**Improvement over the original:** The grouped bar showed only averages. This chart overlays error bars (±1 std dev of finish positions across races), revealing **consistency** alongside pace. A driver with a tight error bar is reliably fast; a wide one is volatile — potentially dangerous when it rains. Sorted by rain average finish position.

In [11]:
eb_long = pd.melt(
    comp.sort_values('Rain_AvgFinish'),
    id_vars=['Driver'],
    value_vars=[
        'Rain_AvgFinish', 'Dry_AvgFinish',
        'Rain_StdFinish',  'Dry_StdFinish'
    ],
    var_name='_key', value_name='_val'
)
eb_long['Condition'] = eb_long['_key'].str.extract(r'^(Rain|Dry)')
eb_long['Metric']    = eb_long['_key'].str.extract(r'_(AvgFinish|StdFinish)$')
eb_long = eb_long.pivot_table(
    index=['Driver','Condition'], columns='Metric', values='_val'
).reset_index()
eb_long.columns.name = None
eb_long['Upper'] = eb_long['AvgFinish'] + eb_long['StdFinish']
eb_long['Lower'] = (eb_long['AvgFinish'] - eb_long['StdFinish']).clip(lower=1)

driver_order_eb = comp.sort_values('Rain_AvgFinish')['Driver'].tolist()
color_scale = alt.Scale(domain=['Rain','Dry'], range=['#1f77b4','#d62728'])

dots = alt.Chart(eb_long).mark_point(filled=True, size=60).encode(
    x=alt.X('Driver:N', sort=driver_order_eb,
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('AvgFinish:Q', title='Avg Finish Position ± 1 std dev',
            scale=alt.Scale(domain=[0, 22])),
    color=alt.Color('Condition:N', scale=color_scale),
    xOffset='Condition:N',
    tooltip=['Driver','Condition',
             alt.Tooltip('AvgFinish:Q', format='.2f', title='Mean Finish'),
             alt.Tooltip('StdFinish:Q', format='.2f', title='Std Dev')]
)

errorbars = alt.Chart(eb_long).mark_errorbar().encode(
    x=alt.X('Driver:N', sort=driver_order_eb),
    y=alt.Y('Lower:Q', title=''),
    y2='Upper:Q',
    color=alt.Color('Condition:N', scale=color_scale),
    xOffset='Condition:N'
)

chart3 = (errorbars + dots).properties(
    title=alt.TitleParams(
        'Finish Position — Mean ± Std Dev (blue=Rain, red=Dry)',
        fontSize=13
    ),
    width=720, height=360
)

chart3

alt.LayerChart(...)

---
## Chart 4 — Lap-by-Lap Position Trajectory

**New insight:** None of the original charts showed *how* positions evolve over the course of a race. This chart takes the **top 6 rain specialists** (lowest `Rain_AvgFinish`) and plots their average position at each lap number in rainy races (solid) vs dry races (dashed). 
This reveals whether a driver's rain advantage comes from an early aggressive charge, a mid-race tyre strategy, or simply attrition of others.

In [14]:
top6_rain = comp.sort_values('Rain_AvgFinish').head(6)['Driver'].tolist()

def lap_trajectory(subset, drivers, condition_label):
    rows = []
    for driver in drivers:
        d = subset[subset['Driver'] == driver][['LapNumber','Position']].dropna()
        d = d[d['LapNumber'] <= 70]  # cap at 70 laps
        lap_avg = d.groupby('LapNumber')['Position'].mean().reset_index()
        lap_avg['Driver']    = driver
        lap_avg['Condition'] = condition_label
        rows.append(lap_avg)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

rain_traj = lap_trajectory(clean[clean['Rainfall'] == True],  top6_rain, 'Rain')
dry_traj  = lap_trajectory(clean[clean['Rainfall'] == False], top6_rain, 'Dry')
traj_df   = pd.concat([rain_traj, dry_traj], ignore_index=True)

# Smooth with 3-lap rolling mean per driver+condition
traj_df = traj_df.sort_values(['Driver','Condition','LapNumber'])
traj_df['PosSmooth'] = (
    traj_df.groupby(['Driver','Condition'])['Position']
    .transform(lambda x: x.rolling(3, min_periods=1, center=True).mean())
)

base = alt.Chart(traj_df).encode(
    x=alt.X('LapNumber:Q', title='Lap Number'),
    y=alt.Y('PosSmooth:Q', title='Avg Position (lower = better)',
            scale=alt.Scale(reverse=True, domain=[1, 20])),
    color=alt.Color('Driver:N', legend=alt.Legend(title='Driver')),
    strokeDash=alt.StrokeDash('Condition:N',
                              scale=alt.Scale(domain=['Rain','Dry'],
                                              range=[[1,0],[6,3]]),
                              legend=alt.Legend(title='Condition')),
    tooltip=['Driver','Condition',
             alt.Tooltip('LapNumber:Q', title='Lap'),
             alt.Tooltip('PosSmooth:Q', format='.1f', title='Avg Position')]
)

chart4 = base.mark_line(strokeWidth=2).properties(
    title=alt.TitleParams(
        'Lap-by-Lap Avg Position — Top 6 Rain Specialists (solid=Rain, dashed=Dry)',
        fontSize=13
    ),
    width=840, height=320
)

chart4

alt.Chart(...)

---
## Chart 5 — Weather Variable Correlation Heatmap

**New insight:** We have five weather variables per lap. This heatmap shows the **Pearson correlation** between every pair of weather variables and also against `LapTimeDiff` (each driver's lap time vs the field median). 
This answers a question none of the previous charts addressed: *which environmental factor is most strongly associated with lap time variation?* A high correlation with `LapTimeDiff` means that variable predicts whether a driver is going faster or slower than the field.

In [13]:
# Compute LapTimeDiff (driver lap time minus median for that lap in that race)
median_lt = (
    clean.groupby(['RaceID','LapNumber'])['LapTime']
    .median()
    .reset_index()
    .rename(columns={'LapTime':'MedianLapTime'})
)
clean2 = clean.merge(median_lt, on=['RaceID','LapNumber'])
clean2['LapTimeDiff'] = clean2['LapTime'] - clean2['MedianLapTime']

weather_vars = ['AirTemp','Humidity','TrackTemp','WindSpeed','LapTimeDiff']
# Rainfall as 0/1
clean2['Rainfall_int'] = clean2['Rainfall'].astype(int)
corr_vars = ['AirTemp','Humidity','TrackTemp','WindSpeed','Rainfall_int','LapTimeDiff']
corr_labels = ['Air Temp','Humidity','Track Temp','Wind Speed','Rainfall','Lap Time Delta']

corr_matrix = clean2[corr_vars].corr()
corr_matrix.index   = corr_labels
corr_matrix.columns = corr_labels

corr_long = (
    corr_matrix.reset_index()
    .melt(id_vars='index', var_name='Variable2', value_name='Correlation')
    .rename(columns={'index': 'Variable1'})
)
corr_long['CorrRounded'] = corr_long['Correlation'].round(3)

heatmap = alt.Chart(corr_long).mark_rect().encode(
    x=alt.X('Variable1:N', title=None, axis=alt.Axis(labelAngle=-35, labelFontSize=11)),
    y=alt.Y('Variable2:N', title=None, axis=alt.Axis(labelFontSize=11)),
    color=alt.Color('Correlation:Q',
                    scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
                    legend=alt.Legend(title='Pearson r')),
    tooltip=['Variable1','Variable2',
             alt.Tooltip('CorrRounded:Q', title='Pearson r')]
)

text_layer = alt.Chart(corr_long).mark_text(fontSize=10).encode(
    x='Variable1:N',
    y='Variable2:N',
    text=alt.Text('CorrRounded:Q', format='.2f'),
    color=alt.condition(
        alt.datum.Correlation > 0.5,
        alt.value('white'), alt.value('black')
    )
)

chart5 = (heatmap + text_layer).properties(
    title=alt.TitleParams(
        'Weather Variable Correlation Matrix (incl. Lap Time Delta)',
        fontSize=13
    ),
    width=420, height=380
)

chart5

alt.LayerChart(...)